<a href="https://colab.research.google.com/github/ziadkhalil04-jpg/ML-internship_test/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ziadkhalil04-jpg/ML-internship_test/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

1-What one row means (Grain):
One row represents daily Search Console performance metrics (⁠clicks⁠, ⁠impressions⁠, ⁠ctr⁠, ⁠position⁠) for a unique combination of ⁠(client_hash_id, page, query, date)⁠.
2-Which table(s) you'll use:
fact_gsc_daily⁠ (or ⁠gsc_search_analytics⁠ subset) joined with ⁠dim_clients⁠ from ⁠FlyRank/internship-warehouse⁠.
3-Which time window:
Training/Feature panel: Mid-panel month ⁠2026-03⁠ (March 1, 2026 to March 31, 2026). Final month ⁠2026-06⁠ is reserved as a sealed test month.
4-What you'd predict or rank (Label / Proxy):
Predicting whether a query-page pair will experience an increase in organic clicks on day t+1 (or predicting next-day CTR / Position rank).
5-One thing you deliberately exclude:
Low-impression noise records where ⁠impressions < 5⁠ or records where client availability ⁠has_gsc_access⁠ is ⁠False⁠.

In [12]:
print(f"Total rows: {len(df_clients)}")
print(f"Unique clients: {df_clients['client_hash_id'].nunique()}")

Total rows: 104
Unique clients: 104


* **Context:**
  * `client_hash_id`, `date`, `page`, `query`
  * *Why:* Used to define row identity, perform joins, and sequence time-series data; not fed directly into model features.

* **Feature:**
  * `impressions_7d_avg`, `clicks_7d_sum`, `avg_position_7d`, `query_length`, `ctr_3d_avg`
  * *Why:* Historical metrics calculated strictly up to day $t-1$ to serve as predictive inputs.

* **Label:**
  * `target_next_clicks`
  * *Why:* The target variable representing search clicks on day $t+1$.

* **Excluded (with reasons):**
  * `is_active` / `access_profile`: Excluded because they are static metadata providing near-zero predictive variance.
  * `raw_response_data`: Excluded to avoid unstructured noise and unnecessary payload complexity.

In [13]:
# 1. Define the four field buckets
context_fields = ['client_hash_id', 'date', 'page', 'query']
feature_fields = ['impressions_7d_avg', 'clicks_7d_sum', 'avg_position_7d', 'query_length', 'ctr_3d_avg']
label_field    = ['target_next_clicks']
excluded_fields = {
    'is_active': 'Constant flag, provides no predictive variance',
    'access_profile': 'Metadata for permissions, not relevant to search performance',
    'raw_response_data': 'Excluded to eliminate noise and payload overhead'
}

# 2. Print field bucket summary
print("=== DATA CONTRACT FIELD BUCKETS ===")
print(f" Context fields ({len(context_fields)}): {context_fields}")
print(f" Feature fields ({len(feature_fields)}): {feature_fields}")
print(f" Label field ({len(label_field)}): {label_field}")
print(f" Excluded fields count: {len(excluded_fields)}")

# 3. Display exclusion reasons
print("\n--- Exclusion Reasons ---")
for field, reason in excluded_fields.items():
    print(f"• {field}: {reason}")

=== DATA CONTRACT FIELD BUCKETS ===
 Context fields (4): ['client_hash_id', 'date', 'page', 'query']
 Feature fields (5): ['impressions_7d_avg', 'clicks_7d_sum', 'avg_position_7d', 'query_length', 'ctr_3d_avg']
 Label field (1): ['target_next_clicks']
 Excluded fields count: 3

--- Exclusion Reasons ---
• is_active: Constant flag, provides no predictive variance
• access_profile: Metadata for permissions, not relevant to search performance
• raw_response_data: Excluded to eliminate noise and payload overhead


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Contract Verification Claims
1. **Grain Verification:** Prove that primary keys uniquely identify rows without duplicate records.
2. **Window & Counts Verification:** Check row counts and verify date range spans for the mid-panel month (`2026-03`).
3. **Availability & Nulls Verification:** Filter available clients (`has_gsc_access IS TRUE` / `is_active IS TRUE`) and check for missing values across required fields.

In [15]:
# -------------------------------------------------------------
# QUERY 1: Grain Check (Uniqueness)
# -------------------------------------------------------------
grain_cols = ['client_hash_id']
duplicates = df_clients.duplicated(subset=grain_cols).sum()
print(f"Query 1 - Grain Uniqueness Check ({grain_cols}):")
print(f"-> Found {duplicates} duplicate rows. (PASS if 0)\n")

# -------------------------------------------------------------
# QUERY 2: Row Counts & Time Window Span
# -------------------------------------------------------------
print(f"Query 2 - Row Counts & Window Check:")
print(f"-> Total Dataset Rows: {len(df_clients)}")

if 'date' in df_clients.columns:
    df_clients['date'] = pd.to_datetime(df_clients['date'])
    march_slice = df_clients[(df_clients['date'] >= '2026-03-01') & (df_clients['date'] <= '2026-03-31')]
    print(f"-> March 2026 Slice Rows: {len(march_slice)}")
    print(f"-> Date Span: {march_slice['date'].min().date()} to {march_slice['date'].max().date()}\n")
else:
    print("-> Date Column: N/A (Dimension table representing static client attributes)\n")

# -------------------------------------------------------------
# QUERY 3: Availability Check (IS TRUE) & Missing Values
# -------------------------------------------------------------
print(f"Query 3 - Availability Filter & Missing Values Check:")
if 'has_gsc_access' in df_clients.columns:
    survived_df = df_clients[df_clients['has_gsc_access'] == True]
    print(f"-> Survived Rows (has_gsc_access IS TRUE): {len(survived_df)} / {len(df_clients)} rows")
    print("-> Missing values in active slice:")
    print(survived_df.isnull().sum())
else:
    print(f"-> Survived Rows (is_active IS TRUE): {len(df_clients[df_clients['is_active'] == True])} / {len(df_clients)}")

Query 1 - Grain Uniqueness Check (['client_hash_id']):
-> Found 0 duplicate rows. (PASS if 0)

Query 2 - Row Counts & Window Check:
-> Total Dataset Rows: 104
-> Date Column: N/A (Dimension table representing static client attributes)

Query 3 - Availability Filter & Missing Values Check:
-> Survived Rows (has_gsc_access IS TRUE): 67 / 104 rows
-> Missing values in active slice:
client_hash_id          0
is_active               0
has_gsc_access          0
has_ga4_access          0
access_profile          0
client_created_date     0
client_updated_date     0
gsc_data_start          7
ga4_data_start         17
dtype: int64


### What can this data never tell you? (Slice Limitations)

* **Unbalanced Historical Depth:** Clients joined at different times have unequal history lengths, making uniform rolling windows (e.g., 30-day averages) incomplete for newer clients.
* **GSC Privacy Thresholds:** Google Search Console anonymizes and omits low-traffic long-tail search queries, leading to potential selection bias towards high-volume queries.
* **Permission Gating Limitations:** Clients with `has_gsc_access == False` cannot be evaluated for organic search metrics, limiting model generalizability to GSC-integrated properties only.

In [16]:
# -------------------------------------------------------------
# Data Limits Verification: Unbalanced History & Access Boundaries
# -------------------------------------------------------------
print("=== DATA LIMITS & BOUNDARIES CHECK ===")

# 1. Access Profile & Integration Boundaries
if 'has_gsc_access' in df_clients.columns and 'has_ga4_access' in df_clients.columns:
    gsc_only = len(df_clients[(df_clients['has_gsc_access'] == True) & (df_clients['has_ga4_access'] == False)])
    no_access = len(df_clients[(df_clients['has_gsc_access'] == False) & (df_clients['has_ga4_access'] == False)])

    print(f"-> Clients with GSC only (Missing GA4 context): {gsc_only}")
    print(f"-> Clients with NO search/analytics access: {no_access}")

# 2. Active vs Inactive Account Disparity
if 'is_active' in df_clients.columns:
    inactive_clients = len(df_clients[df_clients['is_active'] == False])
    active_count = len(df_clients[df_clients['is_active'] == True])
    active_percentage = (active_count / len(df_clients)) * 100

    print(f"-> Inactive clients (Historical gaps): {inactive_clients}")
    print(f"-> Active data coverage ratio: {active_percentage:.1f}%")

=== DATA LIMITS & BOUNDARIES CHECK ===
-> Clients with GSC only (Missing GA4 context): 14
-> Clients with NO search/analytics access: 26
-> Inactive clients (Historical gaps): 20
-> Active data coverage ratio: 71.2%


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/w03_data_contract.ipynb` — then submit your repo URL on the card.

**Done.**